In [ ]:
import pandas as pd

trial = 2
dataset = "train"
filepath = f"trial{trial}_{dataset}_pre.csv"

# Read CSV
df = pd.read_csv(filepath)

# Select cluster column + last 68 columns
cols_to_use = ['cluster'] + df.columns[-68:].tolist()

# Group by cluster and compute mean & std
result = df[cols_to_use].groupby('cluster').agg(['mean', 'std', 'median'])

# Flatten column names
result.columns = ['_'.join(col).strip() for col in result.columns.values]

print(result)


In [ ]:
import pandas as pd

# Load data and select cluster + last 68 columns
df = pd.read_csv(filepath)
cols_to_use = ['cluster'] + df.columns[-68:].tolist()

# Calculate mean and std grouped by cluster
grouped = df[cols_to_use].groupby('cluster').agg(['mean', 'std', 'median'])

# Flatten column multi-index for easier access
grouped.columns = ['_'.join(col).strip() for col in grouped.columns.values]

# Extract columns names without suffix
feature_names = df.columns[-68:].tolist()

### Deviation (Mean)

In [ ]:
# Prepare empty DataFrame to store distances
distances = pd.DataFrame(index=grouped.index)

for feature in feature_names:
    mean_col = feature + '_mean'
    std_col = feature + '_std'
    
    # Mean of cluster 0 for this feature
    mean_0 = grouped.loc[0, mean_col]
    # Std of cluster 0 for this feature
    std_0 = grouped.loc[0, std_col]
    
    # Avoid division by zero
    if std_0 == 0:
        distances[feature + '_distance'] = float('nan')
    else:
        # Calculate distance for all clusters for this feature
        distances[feature + '_distance'] = (grouped[mean_col] - mean_0) / std_0

# Remove the CTX_ prefix and _distance postfix from column names
distances.columns = distances.columns.str.replace('CTX_', '', regex=False)
distances.columns = distances.columns.str.replace('_distance', '', regex=False)
# make all column names lowercase
distances.columns = distances.columns.str.lower()
print(distances)

# get the range of whole distances
whole_distances = distances.values.flatten()
min_distance = whole_distances.min()
max_distance = whole_distances.max()
print(f"Range of whole distances: {min_distance} to {max_distance}")

# Save distances DataFrame to CSV
distances.to_csv(f"brain_img_data_trial{trial}_{dataset}_dv_mean.csv")

### Deviation (Median)

In [ ]:
# Prepare empty DataFrame to store distances
distances = pd.DataFrame(index=grouped.index)

for feature in feature_names:
    # change '_mean' to '_median'
    median_col = feature + '_median'
    std_col = feature + '_std'
    
    # Median of cluster 0 for this feature
    median_0 = grouped.loc[0, median_col]
    # Std of cluster 0 for this feature
    std_0 = grouped.loc[0, std_col]
    
    # Avoid division by zero
    if std_0 == 0:
        distances[feature + '_distance'] = float('nan')
    else:
        # Calculate distance for all clusters for this feature
        distances[feature + '_distance'] = (grouped[median_col] - median_0) / std_0

# Remove the CTX_ prefix and _distance postfix from column names
distances.columns = distances.columns.str.replace('CTX_', '', regex=False)
distances.columns = distances.columns.str.replace('_distance', '', regex=False)
# make all column names lowercase
distances.columns = distances.columns.str.lower()
print(distances)

# get the range of whole distances
whole_distances = distances.values.flatten()
min_distance = whole_distances.min()
max_distance = whole_distances.max()
print(f"Range of whole distances: {min_distance} to {max_distance}")

# Save distances DataFrame to CSV
distances.to_csv(f"brain_img_data_trial{trial}_{dataset}_dv_median.csv")

### Actual Values (Mean)

In [ ]:
# Prepare empty DataFrame to store means
means = pd.DataFrame(index=grouped.index)
for feature in feature_names:
    mean_col = feature + '_mean'
    means[feature] = df[cols_to_use].groupby('cluster')[feature].mean()
# Transform column names
means.columns = means.columns.str.replace('CTX_', '', regex=False)
means.columns = means.columns.str.lower()
print(means)

# get the range of whole means
whole_means = means.values.flatten()
min_mean = whole_means.min()
max_mean = whole_means.max()
print(f"Range of whole means: {min_mean} to {max_mean}")

means.to_csv(f"brain_img_data_trial{trial}_{dataset}_av_means.csv")

### Actual Values (Median)

In [ ]:
# Prepare empty DataFrame to store medians
medians = pd.DataFrame(index=grouped.index)
for feature in feature_names:
    median_col = feature + '_median'
    medians[feature] = df[cols_to_use].groupby('cluster')[feature].median()
# Transform column names
medians.columns = medians.columns.str.replace('CTX_', '', regex=False)
medians.columns = medians.columns.str.lower()
print(medians)
# get the range of whole medians
whole_medians = medians.values.flatten()
min_median = whole_medians.min()
max_median = whole_medians.max()
print(f"Range of whole medians: {min_median} to {max_median}")

medians.to_csv(f"brain_img_data_trial{trial}_{dataset}_av_medians.csv")